In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# "csv": train/test feature tables; "raster": generate the original 2x3-grid split from GeoTIFFs.
DATA_SOURCE = "csv"
TRAIN_CSV = DATA_DIR / "train_70_30_keep_all.csv"
TEST_CSV = DATA_DIR / "test_70_30_keep_all.csv"

# "original_test": follow supplied code; "train_holdout": reserve validation data from training only.
VALIDATION_MODE = "original_test"
RUN_TRAINING = True
RUN_MAP_PREDICTION = False

In [ ]:
FEATURE_NAMES = [
    "Aspect", "Bulk density", "Clay content", "Coarse fragments",
    "Distance to rivers", "Drainage density", "Elevation", "Permeability",
    "Porosity", "LULC", "NDVI", "Plan curvature", "Precipitation",
    "Profile curvature", "Sand content", "Silt content", "Slope", "TWI",
    "Water content at 10 kPa", "Water content at 33 kPa",
    "Water content at 1500 kPa",
]
RASTER_FILES = [
    "rec_aspect.tif", "rec_bulk.tif", "rec_clay.tif", "rec_coar.tif",
    "rec_disriver.tif", "rec_drainage.tif", "rec_elevation.tif",
    "rec_pp_new.tif", "rec_porosity_new.tif", "rec_lulc.tif",
    "rec_ndvi.tif", "rec_plancurv.tif", "rec_precipitation.tif",
    "rec_profilecurv.tif", "rec_sand.tif", "rec_silt.tif",
    "rec_slope.tif", "rec_twi.tif", "rec_water10kpa.tif",
    "rec_water33kpa.tif", "rec_water1500kpa.tif",
]
NUM_FEATURES = 21
IMAGE_SIZE = 25
PATCH_SIZE = 5
assert len(FEATURE_NAMES) == len(RASTER_FILES) == NUM_FEATURES

In [ ]:
def read_raster_stack(data_dir=DATA_DIR):
    """Read aligned, classified rasters in the same order as the source notebook."""
    import rasterio

    layers_data = []
    reference = None
    for filename in RASTER_FILES:
        path = data_dir / filename
        with rasterio.open(path) as src:
            if reference is None:
                reference = (src.shape, src.crs, src.transform)
            elif (src.shape, src.crs, src.transform) != reference:
                raise ValueError(f"Raster grid mismatch: {filename}")
            layers_data.append(src.read(1))
    return np.stack(layers_data, axis=-1)


def source_fill_rules(tensor):
    """Apply the two explicit raster fill rules found in the source notebook."""
    tensor = tensor.copy()
    elev = tensor[:, :, 6]
    perm = tensor[:, :, 7]
    por = tensor[:, :, 8]
    asp = tensor[:, :, 0]
    slope = tensor[:, :, 16]

    fill_soil = (elev != 3257) & (por == 3257)
    por[fill_soil] = 6
    perm[fill_soil] = 1

    # Copy the eastern neighbor only where that neighbor exists.
    fill_terrain = (elev[:, :-1] != 3257) & (asp[:, :-1] == 3257)
    target_asp = asp[:, :-1]
    target_slope = slope[:, :-1]
    target_asp[fill_terrain] = asp[:, 1:][fill_terrain]
    target_slope[fill_terrain] = slope[:, 1:][fill_terrain]
    return tensor


def split_samples_from_rasters(data_dir=DATA_DIR, test_size=0.30, seed=42):
    """Mirror the supplied 2-row by 3-column grid-stratified split."""
    import rasterio

    with rasterio.open(data_dir / "flood_points.tif") as src:
        labels = src.read(1)
        label_shape, label_crs, label_transform = src.shape, src.crs, src.transform
    with rasterio.open(data_dir / RASTER_FILES[0]) as src:
        if (src.shape, src.crs, src.transform) != (label_shape, label_crs, label_transform):
            raise ValueError("Inventory and conditioning rasters are not aligned")

    tensor = source_fill_rules(read_raster_stack(data_dir))
    H, W = labels.shape
    row_edges = [0, H // 2, H]
    col_edges = [0, W // 3, 2 * W // 3, W]
    rng = np.random.default_rng(seed)
    train_parts, test_parts = [], []

    for row in range(2):
        for col in range(3):
            r0, r1 = row_edges[row:row + 2]
            c0, c1 = col_edges[col:col + 2]
            local = labels[r0:r1, c0:c1]
            valid = (local == 0) | (local == 1)
            rr, cc = np.where(valid)
            if len(rr) < 2:
                continue
            rr, cc = rr + r0, cc + c0
            x = tensor[rr, cc, :]
            y = labels[rr, cc].astype(int)
            rc = np.column_stack((rr, cc))
            indices = np.arange(len(y))
            counts = np.bincount(y, minlength=2)
            stratify = y if counts.min() >= 2 and len(y) >= 4 else None
            idx_train, idx_test = train_test_split(
                indices, test_size=test_size, random_state=seed,
                stratify=stratify,
            )
            def frame(idx):
                values = pd.DataFrame(x[idx], columns=[f"feat_{i+1}" for i in range(NUM_FEATURES)])
                values["row"] = rc[idx, 0]
                values["col"] = rc[idx, 1]
                values["target"] = y[idx]
                return values
            train_parts.append(frame(idx_train))
            test_parts.append(frame(idx_test))

    if not train_parts or not test_parts:
        raise ValueError("No valid flood/non-flood points found")
    return pd.concat(train_parts, ignore_index=True), pd.concat(test_parts, ignore_index=True)


def load_input_tables():
    if DATA_SOURCE == "csv":
        return pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
    if DATA_SOURCE == "raster":
        return split_samples_from_rasters()
    raise ValueError("DATA_SOURCE must be 'csv' or 'raster'")

In [ ]:
def clean_feature_table(table, name):
    required = [f"feat_{i+1}" for i in range(NUM_FEATURES)] + ["target"]
    missing = [column for column in required if column not in table.columns]
    if missing:
        raise ValueError(f"{name}: missing columns {missing}")
    x = table[required[:NUM_FEATURES]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    y = pd.to_numeric(table["target"], errors="coerce").to_numpy(dtype=np.float64)

    # Match the source's >2000 NoData exclusion, without changing the remaining observations.
    keep = np.isfinite(y) & np.isfinite(x).all(axis=1) & (x <= 2000).all(axis=1)
    excluded = int((~keep).sum())
    x, y = x[keep], y[keep]
    if x.shape[0] == 0:
        raise ValueError(f"{name}: no samples remain after NoData filtering")
    if not np.isin(y, [0, 1]).all():
        raise ValueError(f"{name}: expected binary target values 0 and 1")
    return x, y.astype(np.float32), excluded

train_table, test_table = load_input_tables()
X_train, y_train, dropped_train = clean_feature_table(train_table, "train")
X_test, y_test, dropped_test = clean_feature_table(test_table, "test")
print(f"Train: {len(y_train)} samples; Test: {len(y_test)} samples")
print(f"Removed invalid rows: train={dropped_train}; test={dropped_test}")

In [ ]:
def encode_classified_features(features):
    """Encode N x 21 integer class codes (1..25) into N x 25 x 25 x 1."""
    features = np.asarray(features, dtype=np.float64)
    if features.ndim != 2 or features.shape[1] != NUM_FEATURES:
        raise ValueError(f"Expected (N, {NUM_FEATURES}), got {features.shape}")
    if not np.isfinite(features).all():
        raise ValueError("Feature matrix contains non-finite values")
    if not np.array_equal(features, np.rint(features)):
        raise ValueError("Features must be integer class codes before one-hot encoding")
    if (features < 1).any() or (features > IMAGE_SIZE).any():
        raise ValueError("Feature class codes must be between 1 and 25")
    n = features.shape[0]
    images = np.zeros((n, IMAGE_SIZE, IMAGE_SIZE, 1), dtype=np.float32)
    row_indices = features.astype(np.int64) - 1
    image_ids = np.arange(n)[:, None]
    columns = np.arange(NUM_FEATURES)[None, :]
    images[image_ids, row_indices, columns, 0] = 1.0
    return images

x_train_img = encode_classified_features(X_train)
x_test_img = encode_classified_features(X_test)

assert x_train_img.shape == (len(y_train), 25, 25, 1)
assert x_test_img.shape == (len(y_test), 25, 25, 1)
assert np.all(x_train_img[:, :, 21:, :] == 0)
assert np.all(x_train_img.sum(axis=(1, 2, 3)) == 21)
print("Encoded shapes:", x_train_img.shape, x_test_img.shape)

In [ ]:
PROJECTION_DIM = 50
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2
NUM_ATTENTION_HEADS = 5       # Exact value in the supplied original source
TRANSFORMER_LAYERS = 1
TRANSFORMER_UNITS = [100, 50]
CLASSIFIER_UNITS = [2048, 1024]
ATTENTION_DROPOUT = 0.2
TRANSFORMER_MLP_DROPOUT = 0.1
FLATTEN_DROPOUT = 0.5
CLASSIFIER_MLP_DROPOUT = 0.5
LEARNING_RATE = 0.001
BATCH_SIZE = 32
MAX_EPOCHS = 30               # Exact maximum in the supplied original source
EARLY_STOPPING_PATIENCE = 15

class PatchExtractor(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        patch_data = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        batch_size = tf.shape(images)[0]
        patch_dim = self.patch_size * self.patch_size * images.shape[-1]
        return tf.reshape(patch_data, (batch_size, -1, patch_dim))

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config


def apply_mlp(inputs, hidden_units, dropout):
    x = inputs
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout)(x)
    return x


def build_vit_model():
    inputs = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1))
    patches = PatchExtractor(PATCH_SIZE, name="patch_extractor")(inputs)
    positions = tf.range(start=0, limit=NUM_PATCHES, delta=1)
    positional_embedding = layers.Embedding(NUM_PATCHES, PROJECTION_DIM)(positions)
    encoded = layers.Dense(PROJECTION_DIM)(patches) + positional_embedding

    for _ in range(TRANSFORMER_LAYERS):
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded)
        attention = layers.MultiHeadAttention(
            num_heads=NUM_ATTENTION_HEADS,
            key_dim=PROJECTION_DIM,
            dropout=ATTENTION_DROPOUT,
        )(x1, x1)
        x2 = layers.Add()([attention, encoded])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = apply_mlp(x3, TRANSFORMER_UNITS, TRANSFORMER_MLP_DROPOUT)
        encoded = layers.Add()([x3, x2])

    x = layers.LayerNormalization(epsilon=1e-6)(encoded)
    x = layers.Flatten()(x)
    x = layers.Dropout(FLATTEN_DROPOUT)(x)
    x = apply_mlp(x, CLASSIFIER_UNITS, CLASSIFIER_MLP_DROPOUT)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs=inputs, outputs=outputs, name="flood_vit")

model = build_vit_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
if VALIDATION_MODE == "original_test":
    fit_x, fit_y = x_train_img, y_train
    validation_data = (x_test_img, y_test)
elif VALIDATION_MODE == "train_holdout":
    fit_x, val_x, fit_y, val_y = train_test_split(
        x_train_img, y_train, test_size=0.15, random_state=RANDOM_SEED,
        stratify=y_train if len(np.unique(y_train)) == 2 else None,
    )
    validation_data = (val_x, val_y)
else:
    raise ValueError("Unknown VALIDATION_MODE")

WEIGHTS_PATH = OUTPUT_DIR / "flood_vit.weights.h5"
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(WEIGHTS_PATH),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=0,
    ),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE),
]

if RUN_TRAINING:
    history = model.fit(
        fit_x, fit_y,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=validation_data,
        callbacks=callbacks,
        verbose=1,
    )
    model.load_weights(str(WEIGHTS_PATH))
else:
    if not WEIGHTS_PATH.exists():
        raise FileNotFoundError(f"Pretrained weights not found: {WEIGHTS_PATH}")
    model.load_weights(str(WEIGHTS_PATH))

In [ ]:
def evaluate_scores(labels, predictions, num_predictors=NUM_FEATURES):
    labels = np.asarray(labels).reshape(-1)
    predictions = np.asarray(predictions).reshape(-1)
    n = len(labels)
    rmse = float(np.sqrt(mean_squared_error(labels, predictions)))
    r2 = float(r2_score(labels, predictions))
    adj_r2 = float(1 - (1 - r2) * (n - 1) / (n - num_predictors - 1)) if n > num_predictors + 1 else float("nan")
    pearson = float(np.corrcoef(labels, predictions)[0, 1]) if np.std(labels) > 0 and np.std(predictions) > 0 else float("nan")
    auc = float(roc_auc_score(labels, predictions)) if len(np.unique(labels)) > 1 else float("nan")
    return {"n": n, "AUC": auc, "RMSE": rmse, "R2": r2,
            "Adjusted_R2": adj_r2, "Pearson_R": pearson}

train_scores = model.predict(x_train_img, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
test_scores = model.predict(x_test_img, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
metrics = pd.DataFrame({
    "Train": evaluate_scores(y_train, train_scores),
    "Test": evaluate_scores(y_test, test_scores),
}).T
print(metrics.to_string(float_format=lambda value: f"{value:.6f}"))

np.save(OUTPUT_DIR / "y_pred_train.npy", train_scores)
np.save(OUTPUT_DIR / "y_pred_test.npy", test_scores)
metrics.to_csv(OUTPUT_DIR / "evaluation_metrics.csv")

In [ ]:
def predict_susceptibility_geotiff(model, data_dir=DATA_DIR,
                                   output_path=OUTPUT_DIR / "flood_vit_susceptibility.tif",
                                   block_size=256, batch_size=256):
    import rasterio
    from rasterio.windows import Window
    from contextlib import ExitStack

    with ExitStack() as stack:
        sources = [stack.enter_context(rasterio.open(data_dir / filename)) for filename in RASTER_FILES]
        reference = sources[0]
        for src in sources[1:]:
            if (src.shape, src.crs, src.transform) != (reference.shape, reference.crs, reference.transform):
                raise ValueError(f"Input grid mismatch: {src.name}")

        profile = reference.profile.copy()
        profile.update(count=1, dtype="float32", nodata=-9999.0, compress="deflate")
        with rasterio.open(output_path, "w", **profile) as dst:
            for r0 in range(0, reference.height, block_size):
                for c0 in range(0, reference.width, block_size):
                    h = min(block_size, reference.height - r0)
                    w = min(block_size, reference.width - c0)
                    window = Window(c0, r0, w, h)
                    # Include the eastern neighbor where available so source-specific fill rules work at block edges.
                    read_width = w + int(c0 + w < reference.width)
                    extended = Window(c0, r0, read_width, h)
                    features = np.stack([src.read(1, window=extended) for src in sources], axis=-1)
                    features = source_fill_rules(features)[:, :w, :]
                    flat = features.reshape(-1, NUM_FEATURES)
                    valid = (np.isfinite(flat).all(axis=1)
                             & np.equal(flat, np.rint(flat)).all(axis=1)
                             & (flat >= 1).all(axis=1)
                             & (flat <= IMAGE_SIZE).all(axis=1))
                    predictions = np.full(flat.shape[0], -9999.0, dtype=np.float32)
                    if np.any(valid):
                        images = encode_classified_features(flat[valid])
                        predictions[valid] = model.predict(images, batch_size=batch_size, verbose=0).reshape(-1)
                    dst.write(predictions.reshape(h, w), 1, window=window)
    return Path(output_path)

if RUN_MAP_PREDICTION:
    result_path = predict_susceptibility_geotiff(model)
    print(f"Wrote susceptibility raster: {result_path}")